In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import time, os, json
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from thop import profile as thop_profile

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU    = torch.device("cpu")

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)
OUTPUT_JSON = "all_quantization_eval.json"

# ── Architecture (must match training) ────────────────────────
def build_model(num_classes=10):
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

# ── Data ───────────────────────────────────────────────────────
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_set    = torchvision.datasets.CIFAR10(root='../../data', train=False,
                                            download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128,
                                           shuffle=False, num_workers=0)

# ══════════════════════════════════════════════════════════════
# Metric helpers
# ══════════════════════════════════════════════════════════════
def evaluate_torch(model, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            all_preds.extend(model(inputs).argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)

def evaluate_onnx(session):
    import onnxruntime
    input_name = session.get_inputs()[0].name
    all_preds, all_labels = [], []
    for inputs, labels in test_loader:
        out = session.run(None, {input_name: inputs.numpy()})[0]
        all_preds.extend(np.argmax(out, axis=1))
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)

def compute_classification_metrics(preds, labels):
    return {
        "accuracy" : float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, average='macro', zero_division=0)),
        "recall"   : float(recall_score(labels, preds, average='macro', zero_division=0)),
        "f1"       : float(f1_score(labels, preds, average='macro', zero_division=0)),
    }

def measure_torch_inference(model, device):
    """Single image + batch-128 timing for PyTorch models."""
    model.eval()
    is_gpu = device.type == "cuda"

    # Single image
    dummy_single = torch.randn(1, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(50): model(dummy_single)
    if is_gpu: torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(500):
            if is_gpu: torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single)
            if is_gpu: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    inf_ms_single = np.mean(times) * 1000

    # Batch 128
    dummy_batch = torch.randn(128, 3, 32, 32, device=device)
    if is_gpu:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(10): model(dummy_batch)
        torch.cuda.synchronize()
        batch_times = []
        with torch.no_grad():
            for _ in range(100):
                start_ev.record()
                model(dummy_batch)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times.append(start_ev.elapsed_time(end_ev))
        inf_ms_batch128 = np.mean(batch_times)
    else:
        # CPU timing (QAT INT8 models run on CPU)
        with torch.no_grad():
            for _ in range(5): model(dummy_batch)
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(20): model(dummy_batch)
        inf_ms_batch128 = (time.perf_counter() - t0) / 20 * 1000

    return {
        "single_img_ms"      : round(inf_ms_single, 4),
        "batch128_ms"        : round(inf_ms_batch128, 4),
        "per_img_ms"         : round(inf_ms_batch128 / 128, 4),
        "throughput_imgs_sec": round(128 / (inf_ms_batch128 / 1000), 1),
        "timing_method"      : "CUDA events + synchronize" if is_gpu else "time.perf_counter (CPU)",
    }

def measure_onnx_inference(session):
    """Wall-clock timing for ONNX Runtime sessions."""
    input_name = session.get_inputs()[0].name
    dummy_np   = np.random.randn(128, 3, 32, 32).astype(np.float32)

    for _ in range(5): session.run(None, {input_name: dummy_np})

    times = []
    for _ in range(100):
        t0 = time.perf_counter()
        session.run(None, {input_name: dummy_np})
        times.append(time.perf_counter() - t0)

    inf_ms_batch128 = np.mean(times) * 1000
    return {
        "single_img_ms"      : None,   # ORT sessions don't support single-image easily
        "batch128_ms"        : round(inf_ms_batch128, 4),
        "per_img_ms"         : round(inf_ms_batch128 / 128, 4),
        "throughput_imgs_sec": round(128 / (inf_ms_batch128 / 1000), 1),
        "timing_method"      : "wall-clock time.perf_counter (ORT)",
    }

def get_flops(model):
    m = model.cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)
    try:
        macs, _ = thop_profile(m, inputs=(dummy,), verbose=False)
        return float(macs * 2 / 1e9)
    except Exception as e:
        print(f"    ⚠ FLOPs failed: {e}")
        return None

def build_result_block(metrics, inf, save_path, total_params, nonzero_params, flops_G):
    size_mb = os.path.getsize(save_path) / 1e6 if os.path.exists(save_path) else None
    return {
        **metrics,
        "params"            : total_params,
        "params_nonzero"    : nonzero_params,
        "sparsity_actual"   : float(1 - nonzero_params / total_params) if total_params else None,
        "size_mb"           : size_mb,
        "flops_G"           : flops_G,
        "inference_ms"      : inf,
    }

def print_result(key, r):
    inf = r["inference_ms"]
    print(f"  Accuracy    : {r['accuracy']:.4f}")
    print(f"  F1          : {r['f1']:.4f}")
    print(f"  Params      : {r['params']:,}  (nonzero: {r['params_nonzero']:,})")
    print(f"  Sparsity    : {r['sparsity_actual']:.3f}")
    print(f"  Size        : {r['size_mb']:.2f} MB")
    print(f"  FLOPs       : {r['flops_G']} GFLOPs")
    print(f"  Batch-128   : {inf['batch128_ms']:.2f} ms")
    print(f"  Throughput  : {inf['throughput_imgs_sec']:.1f} imgs/sec")

# ══════════════════════════════════════════════════════════════
# QAT model loader
# QAT INT8 models need the quantized architecture rebuilt before
# loading weights — you can't use plain build_model() for these.
# ══════════════════════════════════════════════════════════════
def build_qat_model(obs_name):
    from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx
    from torch.ao.quantization import (
        QConfig,
        MovingAverageMinMaxObserver, MovingAveragePerChannelMinMaxObserver,
        MinMaxObserver, PerChannelMinMaxObserver,
    )
    OBSERVER_QCONFIGS = {
        "minmax": QConfig(
            activation=MinMaxObserver.with_args(
                dtype=torch.quint8, qscheme=torch.per_tensor_affine),
            weight=PerChannelMinMaxObserver.with_args(
                dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
        ),
        "moving_avg": QConfig(
            activation=MovingAverageMinMaxObserver.with_args(
                dtype=torch.quint8, qscheme=torch.per_tensor_affine),
            weight=MovingAveragePerChannelMinMaxObserver.with_args(
                dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
        ),
    }
    fp32    = build_model().cpu()
    qconfig = OBSERVER_QCONFIGS[obs_name]
    example = (torch.randn(1, 3, 32, 32),)
    prepared = prepare_qat_fx(fp32, {"": qconfig}, example)
    q_model  = convert_fx(prepared)
    return q_model

# ══════════════════════════════════════════════════════════════
# Model registry
# Each entry: (save_path, model_type, extra_info)
#   model_type: "torch_gpu" | "torch_cpu" | "onnx"
#   extra_info: obs_name for QAT models (needed to rebuild arch)
# ══════════════════════════════════════════════════════════════
MODELS = {
    # ── PTQ ───────────────────────────────────────────────────
    "ptq_dynamic_int8": (
        "__1__saved_models_PTQ/dynamic_ptq_int8.pth",
        "torch_gpu",   # dynamic PTQ runs on GPU (weights stored int8, compute fp32)
        None,
    ),
    "ptq_onnx_minmax": (
        "__1__saved_models_PTQ/onnx_int8_minmax.onnx",
        "onnx",
        None,
    ),
    "ptq_onnx_entropy": (
        "__1__saved_models_PTQ/onnx_int8_entropy.onnx",
        "onnx",
        None,
    ),
    "ptq_onnx_percentile": (
        "__1__saved_models_PTQ/onnx_int8_percentile.onnx",
        "onnx",
        None,
    ),
    # ── QAT ───────────────────────────────────────────────────
    "qat_minmax_constant": (
        "__2__saved_models_QAT/qat_int8_minmax_constant.pth",
        "torch_cpu",   # QAT INT8 converted models run on CPU only
        "minmax",      # obs_name needed to rebuild architecture
    ),
    "qat_minmax_cosine": (
        "__2__saved_models_QAT/qat_int8_minmax_cosine.pth",
        "torch_cpu",
        "minmax",
    ),
    "qat_minmax_step": (
        "__2__saved_models_QAT/qat_int8_minmax_step.pth",
        "torch_cpu",
        "minmax",
    ),
    "qat_moving_avg_constant": (
        "__2__saved_models_QAT/qat_int8_moving_avg_constant.pth",
        "torch_cpu",
        "moving_avg",
    ),
}

# ══════════════════════════════════════════════════════════════
# Main evaluation loop
# ══════════════════════════════════════════════════════════════
all_results = {}

for key, (save_path, model_type, extra) in MODELS.items():
    if not os.path.exists(save_path):
        print(f"⚠  SKIP {key} — not found: {save_path}")
        continue

    print(f"\n{'='*55}")
    print(f"  {key}  [{model_type}]")
    print(f"{'='*55}")

    try:
        # ── Load model ─────────────────────────────────────────
        if model_type == "onnx":
            import onnxruntime
            providers = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                         if "CUDAExecutionProvider" in onnxruntime.get_available_providers()
                         else ["CPUExecutionProvider"])
            session = onnxruntime.InferenceSession(save_path, providers=providers)
            active  = session.get_providers()[0]
            print(f"  Provider: {active}")

            preds, labels = evaluate_onnx(session)
            metrics       = compute_classification_metrics(preds, labels)
            inf           = measure_onnx_inference(session)
            total_params  = 23520842   # same architecture — hardcode or load from baseline json
            nonzero_params = total_params
            flops_G       = 2.623      # same as FP32 baseline (quantization doesn't change FLOPs)

        elif model_type == "torch_gpu":
            model = build_model()
            model.load_state_dict(
                torch.load(save_path, map_location=DEVICE), strict=False)
            model = model.to(DEVICE).eval()

            preds, labels  = evaluate_torch(model, DEVICE)
            metrics        = compute_classification_metrics(preds, labels)
            inf            = measure_torch_inference(model, DEVICE)
            total_params   = sum(p.numel() for p in model.parameters())
            nonzero_params = int(sum((p != 0).sum().item() for p in model.parameters()))
            flops_G        = get_flops(model)

        elif model_type == "torch_cpu":
            # QAT INT8 — rebuild quantized arch then load weights
            obs_name = extra
            model    = build_qat_model(obs_name)
            model.load_state_dict(
                torch.load(save_path, map_location=CPU), strict=False)
            model = model.to(CPU).eval()

            preds, labels  = evaluate_torch(model, CPU)
            metrics        = compute_classification_metrics(preds, labels)
            inf            = measure_torch_inference(model, CPU)
            total_params   = sum(p.numel() for p in model.parameters())
            nonzero_params = total_params   # QAT: no weight zeroing, all params active
            flops_G        = get_flops(model)

        # ── Build and store result ──────────────────────────────
        result = build_result_block(
            metrics, inf, save_path,
            total_params, nonzero_params, flops_G
        )
        result["model_type"] = model_type
        all_results[key] = result
        print_result(key, result)

    except Exception as e:
        import traceback
        print(f"  ✗ FAILED: {e}")
        traceback.print_exc()
        all_results[key] = {"status": "failed", "error": str(e)}

# ── Save JSON ──────────────────────────────────────────────────
with open(OUTPUT_JSON, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\n✓ Saved → {OUTPUT_JSON}")

# ── Summary table ──────────────────────────────────────────────
print(f"\n{'Method':<28} {'Type':<12} {'Acc':>7} {'F1':>7} {'MB':>7} {'Batch ms':>10} {'Imgs/s':>8}")
print("-" * 85)
for k, m in all_results.items():
    if m.get("status") == "failed":
        print(f"{k:<28} {'FAILED':<12}")
        continue
    inf = m["inference_ms"]
    print(f"{k:<28} {m['model_type']:<12} {m['accuracy']:>7.4f} {m['f1']:>7.4f} "
          f"{m['size_mb']:>7.2f} {inf['batch128_ms']:>10.2f} "
          f"{inf['throughput_imgs_sec']:>8.1f}")

e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")



  ptq_dynamic_int8  [torch_gpu]
  Accuracy    : 0.0921
  F1          : 0.0610
  Params      : 23,520,842  (nonzero: 23,520,842)
  Sparsity    : 0.000
  Size        : 94.34 MB
  FLOPs       : 2.623188992 GFLOPs
  Batch-128   : 51.23 ms
  Throughput  : 2498.4 imgs/sec

  ptq_onnx_minmax  [onnx]
  Provider: CUDAExecutionProvider
  Accuracy    : 0.9319
  F1          : 0.9318
  Params      : 23,520,842  (nonzero: 23,520,842)
  Sparsity    : 0.000
  Size        : 94.56 MB
  FLOPs       : 2.623 GFLOPs
  Batch-128   : 63.81 ms
  Throughput  : 2006.0 imgs/sec

  ptq_onnx_entropy  [onnx]
  Provider: CUDAExecutionProvider
  Accuracy    : 0.9319
  F1          : 0.9318
  Params      : 23,520,842  (nonzero: 23,520,842)
  Sparsity    : 0.000
  Size        : 94.56 MB
  FLOPs       : 2.623 GFLOPs
  Batch-128   : 63.85 ms
  Throughput  : 2004.8 imgs/sec

  ptq_onnx_percentile  [onnx]
  Provider: CUDAExecutionProvider
  Accuracy    : 0.9316
  F1          : 0.9315
  Params      : 23,520,842  (nonzero: 23

C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py:206: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared = prepare_qat_fx(fp32, {"": qconfig}, example)
W0429 23:25:51.881000 21100 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
e:\baseline_resnet50_cifar10\env\Lib\site-packa

  Accuracy    : 0.9305
  F1          : 0.9305
  Params      : 0  (nonzero: 0)
  ✗ FAILED: unsupported format string passed to NoneType.__format__

  qat_minmax_cosine  [torch_cpu]


Traceback (most recent call last):
  File "C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py", line 328, in <module>
    print_result(key, result)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
  File "C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py", line 171, in print_result
    print(f"  Sparsity    : {r['sparsity_actual']:.3f}")
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: unsupported format string passed to NoneType.__format__
C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py:206: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torc

...

Traceback (most recent call last):
  File "C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py", line 328, in <module>
    print_result(key, result)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
  File "C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py", line 171, in print_result
    print(f"  Sparsity    : {r['sparsity_actual']:.3f}")
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: unsupported format string passed to NoneType.__format__
C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py:206: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torc

  Accuracy    : 0.9320
  F1          : 0.9319
  Params      : 0  (nonzero: 0)
  ✗ FAILED: unsupported format string passed to NoneType.__format__

  qat_moving_avg_constant  [torch_cpu]


Traceback (most recent call last):
  File "C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py", line 328, in <module>
    print_result(key, result)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
  File "C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py", line 171, in print_result
    print(f"  Sparsity    : {r['sparsity_actual']:.3f}")
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: unsupported format string passed to NoneType.__format__
C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py:206: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torc

  Accuracy    : 0.9302
  F1          : 0.9301
  Params      : 0  (nonzero: 0)
  ✗ FAILED: unsupported format string passed to NoneType.__format__

✓ Saved → all_quantization_eval.json

Method                       Type             Acc      F1      MB   Batch ms   Imgs/s
-------------------------------------------------------------------------------------
ptq_dynamic_int8             torch_gpu     0.0921  0.0610   94.34      51.23   2498.4
ptq_onnx_minmax              onnx          0.9319  0.9318   94.56      63.81   2006.0
ptq_onnx_entropy             onnx          0.9319  0.9318   94.56      63.85   2004.8
ptq_onnx_percentile          onnx          0.9316  0.9315   94.56      64.06   1998.2
qat_minmax_constant          FAILED      
qat_minmax_cosine            FAILED      
qat_minmax_step              FAILED      
qat_moving_avg_constant      FAILED      


Traceback (most recent call last):
  File "C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py", line 328, in <module>
    print_result(key, result)
    ~~~~~~~~~~~~^^^^^^^^^^^^^
  File "C:\Users\Mohammed Elidrissi\AppData\Local\Temp\ipykernel_21100\482390942.py", line 171, in print_result
    print(f"  Sparsity    : {r['sparsity_actual']:.3f}")
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: unsupported format string passed to NoneType.__format__
